In [5]:
from pathlib import Path
import json
from googletrans import Translator
import re

In [6]:
path = Path("./result/reddit_results_2025-08-31.json")

In [7]:
try:
    with path.open(encoding="utf-8") as f:
        data = json.load(f)
except json.JSONDecodeError:
    with path.open(encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

In [8]:
if isinstance(data, dict) and "results" in data:
    posts = data["results"]
elif isinstance(data, dict) and "posts" in data:
    posts = data["posts"]
else:
    posts = data

print(f"type(posts)={type(posts).__name__}, count={len(posts)}")

type(posts)=list, count=100


In [9]:
if isinstance(posts, list) and posts:
    print(posts[0])

{'id': '1n4p8oh', 'fullname': 't3_1n4p8oh', 'subreddit': 'GetNewsme', 'title': 'Hacker Retrieves Critical Crash Data in Tesla Wrongful‑Death Lawsuit', 'selftext': 'A freelance security researcher recovered a crash‑data snapshot that Tesla had said was lost, and the evidence became a key point in a 2019 pedestrian‑fatality wrongful‑death trial.\n\n- The 2019 crash involved a Tesla that struck a pedestrian; investigators seized the damaged infotainment system and autopilot unit for analysis.\n- Tesla initially told the court the vehicle’s “local collision snapshot” was unrecoverable, attributing the loss to clumsy handling rather than intentional concealment.\n- Independent researcher @greentheonly claimed the data remained on the car’s hardware and used forensic methods to extract the snapshot.\n- After the researcher’s findings, Tesla conducted another search and located the previously missing data, again describing the earlier assessment as inaccurate.\n- The recovered data clarified 

In [10]:
# tester
translator = Translator()

text = "Hello World!"
translated_text = translator.translate(text, src="auto", dest='fr').text
translated_text

'Bonjour le monde!'

In [15]:
# googletrans sometimes fails on URLs etc; so a safe translate mechanism is required to check for errors

translator = Translator()

URL_RE = re.compile(r'^(https?://|www\.)', re.I)

def safe_translate(text: str, target='en'):
    if not text or not isinstance(text, str):
        return text
    # skip pure URLs or very short tokens
    if URL_RE.match(text.strip()):
        return text
    # skip if it looks like it's already English
    try:
        det = translator.detect(text)
        if det.lang == target:
            cleaned = text.replace('\n', ' ').replace('\t', ' ')
            return cleaned
    except Exception:
        # detection can also fail — then skip
        pass

    # translate with try-catch block
    try:
        translated = translator.translate(text, src='auto', dest=target).text
        cleaned = translated.replace('\n', ' ').replace('\t', ' ')
        return cleaned
    except Exception:
        # return original on any parsing/network error
        cleaned = text.replace('\n', ' ').replace('\t', ' ')
        return cleaned
 


In [17]:
for post in posts[0:15]:
    if 'title' in post:
        post['title'] = safe_translate(post['title'])
    if 'selftext' in post:
        post['selftext'] = safe_translate(post['selftext'])
        
    print(post)

{'id': '1n4p8oh', 'fullname': 't3_1n4p8oh', 'subreddit': 'GetNewsme', 'title': 'Hacker Retrieves Critical Crash Data in Tesla Wrongful‑Death Lawsuit', 'selftext': 'A freelance security researcher recovered a crash‑data snapshot that Tesla had said was lost, and the evidence became a key point in a 2019 pedestrian‑fatality wrongful‑death trial.  - The 2019 crash involved a Tesla that struck a pedestrian; investigators seized the damaged infotainment system and autopilot unit for analysis. - Tesla initially told the court the vehicle’s “local collision snapshot” was unrecoverable, attributing the loss to clumsy handling rather than intentional concealment. - Independent researcher @greentheonly claimed the data remained on the car’s hardware and used forensic methods to extract the snapshot. - After the researcher’s findings, Tesla conducted another search and located the previously missing data, again describing the earlier assessment as inaccurate. - The recovered data clarified the ve

In [13]:
"""
testing on facebook/bart-large-mnli
https://huggingface.co/facebook/bart-large-mnli
"""

from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")


Device set to use mps:0


In [14]:
sequence_to_classify = "one day I will see the world"
candidate_labels = ['travel', 'cooking', 'dancing']
classifier(sequence_to_classify, candidate_labels)

candidate_labels = ['travel', 'cooking', 'dancing', 'exploration']
classifier(sequence_to_classify, candidate_labels, multi_label=True)


{'sequence': 'one day I will see the world',
 'labels': ['travel', 'exploration', 'dancing', 'cooking'],
 'scores': [0.9945111274719238,
  0.9383884072303772,
  0.005706179421395063,
  0.0018193245632573962]}

In [21]:
text = "Hall oils,  I save in ETFs and Tesla seems to me to be an argor risk of speculation.I know that the sense of ETFs is that I shouldn't worry myself.(Edit: The true geund is ethical nature, but I don't want to spark a fundamental discussion here. And yes, my model 3 was a great car, still consider the stock to be overvalued because it has nothing to do with the cars, but only with the promises. I am aware that TSLA only makes up 1-2%.)  Nevertheless, I would like to know whether there is an ETF or a simply coverable combination that is like MSCI World or FTSE All-World, but without or with very very little TSLA.  Does anyone have any tips?"
text = "microsoft way better"
candidate_labels = ["tesla", "TSLA", "$TSLA"]
classifier(text, candidate_labels, multi_label=True)


{'sequence': 'microsoft way better',
 'labels': ['TSLA', '$TSLA', 'tesla'],
 'scores': [0.04272261634469032, 0.025714365765452385, 0.002312201075255871]}